In [0]:
from pyspark.sql.functions import col, when, monotonically_increasing_id, to_date, datediff, current_date

In [0]:
emp_df = spark.table("01_global_tech_bronze.raw_tables.employee_raw")

transformed_employee_df = (
    emp_df
    .withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy HH:mm"))
    .withColumn("termination_date", to_date(col("termination_date"), "dd-MM-yyyy HH:mm"))
    .withColumn("tenure_days",
        datediff(
            when(col("termination_date").isNotNull(), col("termination_date"))
            .otherwise(current_date()),
            col("hire_date")
        )
    )
    .withColumn("employee_sk", monotonically_increasing_id())
    .withColumnRenamed("is_active", "employee_is_active")
)

transformed_employee_df.write.format("delta").mode("overwrite").saveAsTable("02_global_tech_silver.transformed_tables.transformed_employee")

# display(transformed_employee_df)